In [1]:
%%bash
set -euxo pipefail

pip install -U pip
# Clean out conflicting installs
pip uninstall -y torch torchvision torchaudio bitsandbytes accelerate || true

# Install CUDA-enabled PyTorch for Colab (CUDA 12.1). Fall back to cu118 if needed.
pip install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio \
  || pip install --index-url https://download.pytorch.org/whl/cu118 torch torchvision torchaudio

# Install bitsandbytes (>=0.43.1 to avoid old CUDA-only import paths) + transformers + sentencepiece
pip install "bitsandbytes>=0.43.1" "transformers>=4.44.0" sentencepiece


Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Found existing installation: bitsandbytes 0.47.0
Uninstalling bitsandbytes-0.47.0:
  Successfully uninstalled bitsandbytes-0.47.0
Found existing installation: accelerate 1.10.1
Uninstalling accelerate-1.10.1:
  Successfully uninstalled accelerate-1.10.1
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download.pytorch.org/wh

+ pip install -U pip
+ pip uninstall -y torch torchvision torchaudio bitsandbytes accelerate
+ pip install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.17.1 requires accelerate>=0.21.0, which is not installed.
+ pip install 'bitsandbytes>=0.43.1' 'transformers>=4.44.0' sentencepiece


In [2]:
import torch, bitsandbytes as bnb
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("bitsandbytes:", bnb.__version__)

PyTorch: 2.5.1+cu121
CUDA available: True
bitsandbytes: 0.47.0


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,   # FP16 on GPU
    device_map=None
).to("cuda")                     # or "cpu" if no GPU

chat = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)


prompt = (
    "### User: In one sentence, explain why the sky is blue.\n"
    "### Assistant:"
)

response = chat(
    prompt,
    max_new_tokens=128,
    temperature=0.3,
    do_sample=True,
)[0]["generated_text"]

print(response)

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cuda:0


### User: In one sentence, explain why the sky is blue.
### Assistant: The sky is blue because the sunlight reflects off the Earth's atmosphere, which contains a mixture of gases such as oxygen and nitrogen. These gases absorb some of the blue light, which is then reflected back to our eyes, creating the color blue.
